# Module 7.6 — CI/CD for the Model Lifecycle

**Goal:** Simulate the full automated pipeline — retrain trigger → fine-tune candidate → eval gate → canary deployment → rollback or full rollout. Write the GitHub Actions YAML. Understand what blind spot the static eval gate leaves and how canary closes it.

## 1. Setup

In [ ]:
import os, json, random, time
import numpy as np
import matplotlib.pyplot as plt

SMOKE_TEST = True
random.seed(42)
np.random.seed(42)

os.makedirs('reports', exist_ok=True)
os.makedirs('.github/workflows', exist_ok=True)
print('Config OK')

## 2. Retrain Trigger

In [ ]:
def check_trigger(metrics: dict, new_data_count: int = 0) -> tuple[bool, list]:
    reasons = []
    if metrics.get('mean_confidence', 1.0) < 0.72:
        reasons.append('confidence_drop')
    if metrics.get('escalation_rate', 0.0) > 0.25:
        reasons.append('high_escalation')
    if metrics.get('intent_kl', 0.0) > 0.25:
        reasons.append('distribution_shift')
    if metrics.get('faithfulness_cusum', 0.0) > 5.0:
        reasons.append('cusum_alarm')
    if new_data_count >= 500:
        reasons.append('new_data_available')
    # For scheduled runs: 1 signal sufficient; for alert-driven: 2 required
    fire = len(reasons) >= 2 or (new_data_count >= 500 and len(reasons) >= 1)
    return fire, reasons

# Simulate baseline (healthy)
metrics_healthy = {'mean_confidence': 0.83, 'escalation_rate': 0.12,
                   'intent_kl': 0.08, 'faithfulness_cusum': 1.2}
# Simulate drifted state
metrics_drifted = {'mean_confidence': 0.67, 'escalation_rate': 0.31,
                   'intent_kl': 0.33, 'faithfulness_cusum': 6.1}

fire_h, reasons_h = check_trigger(metrics_healthy, new_data_count=100)
fire_d, reasons_d = check_trigger(metrics_drifted, new_data_count=100)

print(f'Healthy  — trigger={fire_h}  reasons={reasons_h}')
print(f'Drifted  — trigger={fire_d}  reasons={reasons_d}')

# New data alone is sufficient for scheduled run
fire_nd, reasons_nd = check_trigger(metrics_healthy, new_data_count=600)
print(f'New data — trigger={fire_nd}  reasons={reasons_nd}')

## 3. Fine-Tune Candidate (Stub)

In [ ]:
def finetune_stub(scenario: str = 'good') -> dict:
    if SMOKE_TEST:
        print('SMOKE_TEST=True — simulating fine-tune run.')
        print('  Base model:  Qwen2.5-1.5B-Instruct')
        print('  LoRA rank:   16')
        print('  Epochs:      3')
        print('  New tickets: 600')
        time.sleep(0.1)  # simulate training time
    if scenario == 'good':
        return {
            'rouge_l':      0.493,
            'intent_f1':    0.902,
            'no_citation':  0.028,
            'per_cat_f1': {
                'billing_dispute': 0.91,
                'technical_bug':   0.89,
                'account_access':  0.93,
                'general_inquiry': 0.88,
                'refund':          0.86,
            },
        }
    elif scenario == 'regression':
        return {
            'rouge_l':      0.455,
            'intent_f1':    0.871,
            'no_citation':  0.067,
            'per_cat_f1': {
                'billing_dispute': 0.90,
                'technical_bug':   0.72,   # hidden category regression
                'account_access':  0.94,
                'general_inquiry': 0.89,
                'refund':          0.85,
            },
        }
    elif scenario == 'sneaky':  # passes overall F1 but fails per-category
        return {
            'rouge_l':      0.480,
            'intent_f1':    0.891,
            'no_citation':  0.031,
            'per_cat_f1': {
                'billing_dispute': 0.96,
                'technical_bug':   0.81,   # 8pp drop — over the 2pp threshold
                'account_access':  0.95,
                'general_inquiry': 0.90,
                'refund':          0.88,
            },
        }

PROD_METRICS = {
    'rouge_l':     0.470,
    'intent_f1':   0.880,
    'no_citation': 0.030,
    'per_cat_f1': {
        'billing_dispute': 0.89,
        'technical_bug':   0.89,
        'account_access':  0.91,
        'general_inquiry': 0.87,
        'refund':          0.85,
    },
}

print('Prod metrics:', PROD_METRICS)
print('Good candidate:', finetune_stub('good'))

## 4. Eval Gate

In [ ]:
def eval_gate(candidate: dict, prod: dict) -> tuple[bool, dict]:
    checks = {
        'rouge_l':          candidate['rouge_l']   >= prod['rouge_l']   - 0.01,
        'intent_f1':        candidate['intent_f1'] >= prod['intent_f1'] - 0.01,
        'no_citation_rate': candidate['no_citation'] <= 0.05,
    }
    cat_checks = {
        cat: candidate['per_cat_f1'].get(cat, 0) >= prod['per_cat_f1'].get(cat, 0) - 0.02
        for cat in prod['per_cat_f1']
    }
    checks['per_category'] = all(cat_checks.values())
    passed = all(checks.values())
    return passed, {**checks, 'per_cat_detail': cat_checks}

cand_good       = finetune_stub('good')
cand_regression = finetune_stub('regression')
cand_sneaky     = finetune_stub('sneaky')

for label, cand in [('GOOD', cand_good), ('REGRESSION', cand_regression), ('SNEAKY', cand_sneaky)]:
    passed, detail = eval_gate(cand, PROD_METRICS)
    print(f'Candidate {label}: {"PASS" if passed else "FAIL"}')
    for k, v in detail.items():
        if k != 'per_cat_detail':
            print(f'  {k}: {"pass" if v else "FAIL"}')
    if not detail['per_category']:
        for cat, ok in detail['per_cat_detail'].items():
            if not ok:
                print(f'    per_cat FAIL: {cat} candidate={cand["per_cat_f1"][cat]} '
                      + f'prod={PROD_METRICS["per_cat_f1"][cat]}')
    print()

## 5. Canary Deployment Simulator

In [ ]:
CANARY_FRACTION        = 0.05
ROLLBACK_ESC_DELTA     = 0.03
ROLLBACK_FAITH_DELTA   = 0.05

def simulate_canary_metrics(candidate_scenario: str, hours: int = 24) -> dict:
    if candidate_scenario == 'good':
        return {'escalation_rate': 0.11, 'faithfulness_p50': 0.77,
                'latency_p50_ms': 1380, 'n_requests': hours * 42}
    elif candidate_scenario == 'degraded':
        return {'escalation_rate': 0.34, 'faithfulness_p50': 0.58,
                'latency_p50_ms': 2100, 'n_requests': hours * 42}
    elif candidate_scenario == 'marginal':
        return {'escalation_rate': 0.15, 'faithfulness_p50': 0.71,
                'latency_p50_ms': 1450, 'n_requests': hours * 42}

PROD_LIVE = {'escalation_rate': 0.12, 'faithfulness_p50': 0.74,
             'latency_p50_ms': 1400}

def should_rollback(canary: dict, prod: dict) -> tuple[bool, str]:
    esc_delta   = canary['escalation_rate'] - prod['escalation_rate']
    faith_delta = prod['faithfulness_p50']  - canary['faithfulness_p50']
    if esc_delta > ROLLBACK_ESC_DELTA:
        return True, f'escalation degraded by {esc_delta:.3f} (threshold {ROLLBACK_ESC_DELTA})'
    if faith_delta > ROLLBACK_FAITH_DELTA:
        return True, f'faithfulness degraded by {faith_delta:.3f} (threshold {ROLLBACK_FAITH_DELTA})'
    return False, 'ok — promote to full rollout'

for scenario in ['good', 'degraded', 'marginal']:
    canary_m = simulate_canary_metrics(scenario)
    rollback, reason = should_rollback(canary_m, PROD_LIVE)
    print(f'Canary ({scenario:<9}): rollback={rollback}  reason={reason}')

## 6. Full Pipeline — End-to-End Run

In [ ]:
def run_pipeline(metrics_24h: dict, new_data_count: int,
                 candidate_scenario: str = 'good') -> dict:
    log = []

    # Stage 1: Trigger
    fire, reasons = check_trigger(metrics_24h, new_data_count)
    log.append(f'[trigger] fire={fire} reasons={reasons}')
    if not fire:
        return {'outcome': 'no_retrain', 'log': log}

    # Stage 2: Fine-tune
    log.append('[finetune] starting QLoRA fine-tune...')
    candidate = finetune_stub(candidate_scenario)
    log.append(f'[finetune] done. rouge_l={candidate["rouge_l"]} intent_f1={candidate["intent_f1"]}')

    # Stage 3: Eval gate
    passed, detail = eval_gate(candidate, PROD_METRICS)
    log.append(f'[eval_gate] passed={passed}')
    if not passed:
        failed = [k for k, v in detail.items() if not v and k != 'per_cat_detail']
        log.append(f'[eval_gate] FAILED checks: {failed}')
        return {'outcome': 'rejected', 'log': log}

    # Stage 4: Canary
    log.append(f'[canary] routing {int(CANARY_FRACTION*100)}% traffic to candidate...')
    canary_m = simulate_canary_metrics(candidate_scenario)
    rollback, reason = should_rollback(canary_m, PROD_LIVE)
    log.append(f'[canary] 24h metrics: esc={canary_m["escalation_rate"]} '
               + f'faith={canary_m["faithfulness_p50"]}')

    if rollback:
        log.append(f'[rollback] TRIGGERED: {reason}')
        return {'outcome': 'rollback', 'log': log}

    log.append('[promote] candidate promoted to 100% traffic.')
    return {'outcome': 'promoted', 'log': log}

scenarios = [
    ('drifted', 600, 'good',      'Ideal: drift + new data + good candidate'),
    ('drifted', 600, 'regression','Bad candidate: eval gate catches it'),
    ('drifted', 600, 'sneaky',    'Sneaky: passes overall F1, per-category gate catches it'),
    ('drifted', 600, 'degraded',  'Passes eval gate but degrades live: canary catches it'),
]

for metrics_key, nd, cand_scen, desc in scenarios:
    metrics = metrics_drifted if metrics_key == 'drifted' else metrics_healthy
    result = run_pipeline(metrics, nd, cand_scen)
    print(f'--- {desc} ---')
    for line in result['log']:
        print(f'  {line}')
    print(f'  OUTCOME: {result["outcome"]}\n')

## 7. Pipeline Outcomes Chart

In [ ]:
outcome_data = [
    ('Good candidate', 'promoted'),
    ('Regression candidate', 'rejected'),
    ('Sneaky regression', 'rejected'),
    ('Passes gate, fails live', 'rollback'),
]

outcomes_count = {'promoted': 0, 'rejected': 0, 'rollback': 0, 'no_retrain': 0}
for _, o in outcome_data:
    outcomes_count[o] += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pipeline stage flow (simulated pass rates)
ax = axes[0]
stages    = ['Triggered', 'Eval gate\npassed', 'Canary\nstable', 'Promoted']
pass_rates = [1.0, 0.70, 0.90, 0.90]
bar_vals   = [1.0, 0.70, 0.63, 0.57]
ax.bar(stages, bar_vals, color=['steelblue','green','darkorange','limegreen'], alpha=0.8)
ax.set_ylabel('Fraction of retrain runs reaching this stage')
ax.set_title('Pipeline Stage Pass Rates (illustrative)')
for i, (v, r) in enumerate(zip(bar_vals, pass_rates)):
    ax.text(i, v + 0.01, f'{int(v*100)}%', ha='center', fontsize=9)
ax.set_ylim(0, 1.15)

# Outcome breakdown
ax = axes[1]
filtered = {k: v for k, v in outcomes_count.items() if v > 0}
colors   = {'promoted': 'limegreen', 'rejected': 'coral', 'rollback': 'darkorange'}
ax.pie([filtered[k] for k in filtered],
       labels=[k.capitalize() for k in filtered],
       colors=[colors.get(k,'grey') for k in filtered],
       autopct='%1.0f%%', startangle=90)
ax.set_title('Pipeline Outcomes (simulated)')

plt.tight_layout()
plt.savefig('cicd_outcomes.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: cicd_outcomes.png')

## 8. GitHub Actions Pipeline YAML

In [ ]:
ACTIONS_YAML = (
    'name: DeskMate Model CI/CD\n'
    '\n'
    'on:\n'
    '  schedule:\n'
    '    - cron: "0 2 * * 1"   # weekly Monday 2 AM UTC\n'
    '  workflow_dispatch:\n'
    '    inputs:\n'
    '      trigger_reason:\n'
    '        description: Why retrain?\n'
    '        required: false\n'
    '        default: scheduled\n'
    '\n'
    'jobs:\n'
    '  model-cicd:\n'
    '    runs-on: ubuntu-latest\n'
    '    steps:\n'
    '      - uses: actions/checkout@v4\n'
    '\n'
    '      - name: Set up Python\n'
    '        uses: actions/setup-python@v5\n'
    '        with:\n'
    '          python-version: "3.11"\n'
    '\n'
    '      - name: Install dependencies\n'
    '        run: pip install -q transformers peft datasets rouge-score\n'
    '\n'
    '      - name: Check retrain trigger\n'
    '        id: trigger\n'
    '        run: |\n'
    '          python scripts/check_trigger.py --output /tmp/trigger.json\n'
    '          echo "should_retrain=$(python -c \"import json; '
    'print(json.load(open(\'\'/tmp/trigger.json\'\'))[\'fire\'])\")" >> $GITHUB_OUTPUT\n'
    '\n'
    '      - name: Fine-tune candidate\n'
    '        if: steps.trigger.outputs.should_retrain == \'True\'\n'
    '        run: |\n'
    '          python scripts/finetune.py \\\n'
    '            --base Qwen/Qwen2.5-1.5B-Instruct \\\n'
    '            --data data/tickets_new.jsonl \\\n'
    '            --output checkpoints/candidate/\n'
    '\n'
    '      - name: Run eval gate\n'
    '        id: eval\n'
    '        if: steps.trigger.outputs.should_retrain == \'True\'\n'
    '        run: |\n'
    '          python scripts/eval_gate.py \\\n'
    '            --candidate checkpoints/candidate/ \\\n'
    '            --prod checkpoints/prod/ \\\n'
    '            --gold data/gold/ \\\n'
    '            --output /tmp/gate.json\n'
    '          echo "passed=$(python -c \"import json; '
    'print(json.load(open(\'\'/tmp/gate.json\'\'))[\'passed\'])\")" >> $GITHUB_OUTPUT\n'
    '\n'
    '      - name: Deploy to canary (5%)\n'
    '        if: steps.eval.outputs.passed == \'True\'\n'
    '        run: python scripts/deploy_canary.py --fraction 0.05\n'
    '\n'
    '      - name: Monitor canary 24h\n'
    '        if: steps.eval.outputs.passed == \'True\'\n'
    '        run: python scripts/monitor_canary.py --wait 86400 --rollback-on-fail\n'
    '\n'
    '      - name: Promote to 100%\n'
    '        if: steps.eval.outputs.passed == \'True\'\n'
    '        run: python scripts/deploy_canary.py --fraction 1.0\n'
    '\n'
    '      - name: Notify on failure\n'
    '        if: failure()\n'
    '        run: python scripts/notify.py --message "Model CI/CD pipeline failed"\n'
)

with open('.github/workflows/model_cicd.yml', 'w') as f:
    f.write(ACTIONS_YAML)

print('Written: .github/workflows/model_cicd.yml')
print()
print(ACTIONS_YAML[:600], '...')

## 9. Checkpoint

In [ ]:
checkpoint = (
    'Your eval gate compares the new model only against the static gold set.\n'
    'What blind spot does this leave, and how does canary deployment address it?\n\n'
    'BLIND SPOT:\n'
    '  The static gold set reflects the ticket distribution at collection time.\n'
    '  Two failure modes the gold set cannot catch:\n'
    '  1. Distribution gap: if live traffic has shifted (KL detected), new ticket types\n'
    '     are not represented in the gold set. A model that regresses on new categories\n'
    '     while improving on old ones can still pass the gate.\n'
    '  2. Low-frequency tails: the gold set is small (hundreds of examples). Rare\n'
    '     failure modes that appear in <1% of live traffic may not appear at all.\n\n'
    'HOW CANARY ADDRESSES IT:\n'
    '  Canary routes 5% of REAL live traffic to the candidate for 24 hours.\n'
    '  This exposes it to the actual current distribution - including new ticket types.\n'
    '  If the candidate degrades (escalation rises, faithfulness drops), the monitor\n'
    '  detects the regression and triggers rollback before 95% of traffic is affected.\n\n'
    'ANALOGY:\n'
    '  Eval gate = unit test (known cases, controlled inputs).\n'
    '  Canary    = integration test on live traffic (unknown cases, real distribution).\n'
    '  Both are necessary. Neither alone is sufficient.'
)
print(checkpoint)

## 10. Save Report

In [ ]:
good_result       = run_pipeline(metrics_drifted, 600, 'good')
regression_result = run_pipeline(metrics_drifted, 600, 'regression')
sneaky_result     = run_pipeline(metrics_drifted, 600, 'sneaky')
degraded_result   = run_pipeline(metrics_drifted, 600, 'degraded')

report = [
    '# CI/CD Model Lifecycle Report\n',
    f'Smoke test: {SMOKE_TEST}\n',
    '## Pipeline Scenario Outcomes',
    '',
    f'Good candidate:              {good_result["outcome"]}',
    f'Regression candidate:        {regression_result["outcome"]}',
    f'Sneaky per-category failure: {sneaky_result["outcome"]}',
    f'Passes gate, fails live:     {degraded_result["outcome"]}',
    '',
    '## Eval Gate Thresholds',
    '',
    '  ROUGE-L:           candidate >= prod - 0.01',
    '  Intent macro-F1:   candidate >= prod - 0.01',
    '  Per-category F1:   no category drops > 2pp',
    '  No-citation rate:  candidate <= 5%',
    '',
    '## Canary Thresholds',
    '',
    f'  Fraction: {int(CANARY_FRACTION*100)}% live traffic for 24h',
    f'  Rollback if escalation degrades by > {ROLLBACK_ESC_DELTA}',
    f'  Rollback if faithfulness degrades by > {ROLLBACK_FAITH_DELTA}',
    '',
    '## Checkpoint',
    '',
    'Blind spot: gold set does not reflect live distribution shift or new ticket types.',
    'Canary closes it: 24h on real traffic catches regressions on new categories.',
    'Eval gate = unit test. Canary = integration test on live distribution.',
]

with open('reports/cicd_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/cicd_report.md')
print('\n=== Module 7.6 Complete ===')
print('=== Phase 7 Complete ===')